# Ceasul Cuantic al Iașului
## Nivelul 4 — Cheia Asachi

> ### Campusul TUIASI, Facultatea de Automatică și Calculatoare, ora 11:30
>
> Mesajul de la Copou v-a adus aici. **Nodul Facultății** păstrează o ghicitoare care spune unde a ascuns Umbra cheia de repornire.
>
> Dar nodul o trimite doar **criptată**, cu o cheie secretă pe care o construiți împreună, prin protocolul cuantic **BB84**.
>
> Problema: spre Facultate duc două linii, **A** și **B**. Pe una dintre ele ascultă Umbra. Trebuie s-o prindeți în flagrant.

### Ce veți învăța în acest nivel

- cum se criptează un mesaj cu o cheie secretă (**XOR**);
- cum două părți construiesc o cheie secretă cu qubiți — protocolul **BB84** (Bennett și Brassard, 1984);
- de ce un spion care măsoară qubiții **lasă urme** și cum îl detectați cu probabilități.

In [ ]:
# Rulați această celulă prima. Ea încarcă Qiskit și uneltele provocării.
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram
from ceas_cuantic import *

ECHIPA = "Numele echipei"   # COMPLETEAZĂ AICI — același nume în toate notebook-urile
COD_ECHIPA = "XXXX-XXX"     # COMPLETEAZĂ AICI — codul primit de la organizatori
start(ECHIPA, COD_ECHIPA)

## 1. Criptarea cu XOR

Dacă Alice și Bob au aceeași cheie secretă de biți, Alice criptează mesajul m bit cu bit cu cheia k: $c = m \oplus k$ (XOR: 0⊕0 = 0, 0⊕1 = 1, 1⊕0 = 1, 1⊕1 = 0). Bob decriptează mesajul criptat c la fel: $c \oplus k = m$. Dacă cheia e aleatoare și folosită o singură dată, nimeni altcineva nu poate citi mesajul.

Problema grea este alta: **cum ajunge cheia la amândoi fără să o vadă nimeni?**

## 2. Două baze

Un bit poate fi pus pe un qubit în două feluri (**baze**):

| | bit 0 | bit 1 | cum pregătim |
| --- | --- | --- | --- |
| baza **Z** (standard) | $\lvert 0\rangle$ | $\lvert 1\rangle$ | X dacă bitul e 1 |
| baza **X** (diagonală) | $\lvert +\rangle$ | $\lvert -\rangle$ | X dacă bitul e 1, apoi H |

Ca să citească în baza X, Bob aplică H înainte de măsurare. **Dacă Bob alege baza greșită, obține un bit complet aleator.**

## 3. Protocolul BB84

1. Alice alege biți și baze aleatoare și trimite câte un qubit pentru fiecare bit.
2. Bob alege, tot aleator, baza în care măsoară fiecare qubit.
3. La final, își spun public **doar bazele** (nu biții!) și păstrează pozițiile unde bazele coincid. Aceasta este **cernerea** (*sifting*). Rămân cam jumătate din biți: cheia.

Întâi exersați în laborator, unde jucați ambele roluri. Apoi vă conectați la adevăratul nod al Facultății, care rulează pe serverul organizatorilor: acolo voi sunteți doar Bob, iar biții lui Alice rămân secreți.

<div class="alert alert-block alert-success">
<b>Sarcina 4A — Prima cheie BB84 (100 de puncte · clopotul 4)</b>

Completați cele două funcții, folosind tabelul cu bazele de mai sus:
<ul>
<li><code>alice_pregateste(bit, baza)</code> — returnează un circuit cu 1 qubit care codifică bitul în baza dată (<code>"Z"</code> sau <code>"X"</code>);</li>
<li><code>bob_alege_baza(qc, baza)</code> — dacă baza e <code>"X"</code>, adaugă H pe qubit; apoi returnează circuitul, gata de măsurat.</li>
</ul>
Cernerea (<code>cerne</code>) și calculul ratei erorilor (<code>rata_erorilor</code>) vi le dăm gata făcute.<br>
Apoi rulați protocolul pe <code>canal_test</code>, o linie de laborator fără Umbra: cheile lui Alice și Bob trebuie să fie identice.
</div>

In [ ]:
def alice_pregateste(bit, baza):
    qc = QuantumCircuit(1)
    # COMPLETEAZĂ AICI
    return qc


def bob_alege_baza(qc, baza):
    # COMPLETEAZĂ AICI
    return qc

Funcția de mai jos rulează tot protocolul pe linia de test. Citiți-o: folosește funcțiile voastre, funcția gata făcută `cerne` și `masoara_biti`, care măsoară fiecare qubit o singură dată.

In [ ]:
def ruleaza_bb84(n, canal):
    biti_alice = genereaza_biti(n)
    baze_alice = genereaza_baze(n)
    baze_bob = genereaza_baze(n)

    circuite = []
    for i in range(n):
        qc = alice_pregateste(biti_alice[i], baze_alice[i])   # Alice pregătește
        qc = canal(qc)                                         # qubitul călătorește
        circuite.append(bob_alege_baza(qc, baze_bob[i]))       # Bob își alege baza
    biti_bob = masoara_biti(circuite)                          # Bob măsoară

    cheie_alice = cerne(biti_alice, baze_alice, baze_bob)
    cheie_bob = cerne(biti_bob, baze_alice, baze_bob)
    return cheie_alice, cheie_bob


cheie_alice, cheie_bob = ruleaza_bb84(64, canal_test)
print(f"Din 64 de qubiți au rămas {len(cheie_alice)} biți de cheie.")
print("Alice:", cheie_alice[:20])
print("Bob:  ", cheie_bob[:20])

In [ ]:
verifica_4A(alice_pregateste, bob_alege_baza, cheie_alice, cheie_bob)

## 4. Umbra pe fir

Acum vă conectați la nodul Facultății. Funcția `conecteaza_la_facultate(linie, n, bob_alege_baza)` face, pentru voi, pașii protocolului:

1. Facultatea (Alice) trimite `n` qubiți pe linia aleasă; Bob îi măsoară în baze aleatoare, cu funcția voastră.
2. Alice anunță public bazele; păstrați pozițiile cu aceeași bază.

Umbra interceptează fiecare qubit de pe linia ei, îl măsoară într-o bază aleasă la întâmplare și trimite mai departe ce a văzut. Nu are cum să facă altfel: un qubit necunoscut **nu poate fi copiat** (teorema „no-cloning”). Când Umbra ghicește greșit baza, strică qubitul, iar Bob ajunge să aibă biți diferiți de ai lui Alice.

3. **Primii 40 de biți** ai cheii se compară public, ca să aflați **rata erorilor**, și apoi se aruncă. Restul formează cheia.

<div class="alert alert-block alert-success">
<b>Sarcina 4B — Linia spionată (80 de puncte)</b>

<ol>
<li>Conectați-vă la Facultate cu 400 de qubiți pe linia A și pe linia B și comparați ratele erorilor.</li>
<li>Alegeți linia sigură.</li>
</ol>
<b>Discutați:</b> pe linia spionată rata iese în jur de 25%. Puteți explica, cu probabilități, de unde vine 1/4?
</div>

In [ ]:
legatura_A = conecteaza_la_facultate("A", 400, bob_alege_baza)
legatura_B = conecteaza_la_facultate("B", 400, bob_alege_baza)

linia_sigura = ""   # COMPLETEAZĂ AICI: "A" sau "B", rata erorilor se acceseaza punand .rata dupa legatura_A sau legatura_B
verifica_4B(linia_sigura)

<div class="alert alert-block alert-success">
<b>Sarcina 4C — Ghicitoarea (70 de puncte · clopotul 5)</b>

Cereți ghicitoarea pe legătura sigură: nodul Facultății o criptează cu cheia lui Alice.<br>
Completați o singură linie din <code>descifreaza(cifru, cheie)</code>: XOR între bitul criptat și bitul de cheie (în Python, <code>c ^ k</code>). Apoi citiți ghicitoarea cu cheia voastră (a lui Bob).<br><br>
Curioși? Încercați și cu legătura spionată: ce iese?<br>
Blocați de tot? <code>kit_de_salvare(4)</code> vă dă ghicitoarea, dar sarcina 4C nu mai aduce puncte și clopotul 5 rămâne stins.
</div>

In [ ]:
legatura = legatura_A if linia_sigura == "A" else legatura_B
cifru = cere_ghicitoarea(legatura)
print("Mesaj criptat:", "".join(map(str, cifru[:48])), "...")


def descifreaza(cifru, cheie):
    biti = []
    # COMPLETEAZA AICI: pentru fiecare bit c din cifru și bit k din cheie, adaugă la lista biti rezultatul XOR dintre c și k
    return biti


ghicitoare = biti_in_text(descifreaza(cifru, legatura.cheie))
print("Ghicitoarea:", ghicitoare)
verifica_4C(ghicitoare)

Ghicitoarea vorbește în limbajul biților. **B0, B1, B2** sunt cei trei biți ai unui loc din Iași. Păstrați-o: o veți transforma într-un oracol cuantic la nivelul 5.

<div class="alert alert-block alert-warning">
<b>Provocare extra — Câți biți verificăm? (+50 de puncte)</b>

Dacă Umbra interceptează fiecare qubit, fiecare bit comparat are șansa 3/4 să nu arate nicio eroare.<br>
Care este cel mai mic număr k de biți comparați pentru care Umbra este prinsă cu o probabilitate de cel puțin 99%?

Folositi np.ceil pentru rotunjirea superioara si np.log pentru logaritm
</div>


In [ ]:
k = 0   # INLOCUIESTE AICI
verifica_extra4(k)

<div class="alert alert-block alert-info">
<b>La AC studiezi asta</b>

Ați construit o cheie secretă protejată de legile fizicii. La AC studiați criptografia, securitatea rețelelor și criptografia post-cuantică: algoritmi clasici gândiți să reziste chiar și calculatoarelor cuantice de mâine.
</div>

> ### Umbra a fost văzută
>
> Clopotele 4 și 5 sună unul după altul. Umbra a lăsat urme pe linie, iar ghicitoarea e a voastră. Mai trebuie doar s-o dezlegați.

## Sfârșitul nivelului 4

Rulați celula de mai jos ca să vedeți turnul echipei.

Când sunteți gata, deschideți **Nivelul_5_Ascunzatoarea.ipynb**.

In [ ]:
turn()